In [1]:
#Cargar librerias generales y documento para el historial


import pandas as pd
import numpy as np
from pathlib import Path
import re
import time
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

#prereq = "para trabajar pre req progv22025-09-15095342_procesado.xlsx"
#historial = "detalle matricula cohorte 2019.xlsx"
historial = "resultados_prerrequisitos_v6_cadenas_2026-02-02175405.csv"
asig_plan = "asignaturas en plan de estudio v2.xlsx"
                                

In [2]:
## Transformacion de datos: Emular pasos de Power Query

import pandas as pd
import numpy as np

def transformar_datos(df: pd.DataFrame,
                      asig_en_planes_estudio: pd.DataFrame | None = None,
                      drop_error_rows: bool = False) -> pd.DataFrame:
    """
    Transforma el DataFrame siguiendo los pasos de Power Query proporcionados.
    
    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame de entrada.
    asig_en_planes_estudio : pd.DataFrame | None
        DataFrame externo con al menos las columnas ['MatCrso','En_plan_de_estudio'] para combinar.
    drop_error_rows : bool
        Si True, emula Table.RemoveRowsWithErrors eliminando filas con errores de conversión
        en columnas de prerequisitos (Nota/Intentos/Nivel). Por defecto False.

    Retorna
    -------
    pd.DataFrame
        DataFrame transformado.
    """
    d = df.copy()

    # ---------------------------------------------
    # Helpers
    # ---------------------------------------------
    def cols_exist(cols):
        return [c for c in cols if c in d.columns]

    def replace_dot_with_comma(cols):
        # Emula ReplaceValue("." , ",") sólo en columnas de texto (object/string)
        for c in cols_exist(cols):
            if pd.api.types.is_string_dtype(d[c]) or d[c].dtype == object:
                d[c] = d[c].str.replace(".", ",", regex=False)

    def to_number(col, prefer_int=False):
        """Convierte a número tolerando coma/punto y valores string; devuelve float/Int64 (nullable)."""
        if col not in d.columns:
            return
        s = d[col]
        if s.dtype.kind in "biufc":  # ya numérica
            if prefer_int:
                d[col] = pd.to_numeric(s, errors="coerce").astype("Int64")
            else:
                d[col] = pd.to_numeric(s, errors="coerce")
            return
        # normalizar separador decimal para parseo
        s_norm = s.astype(str).str.replace(",", ".", regex=False)
        num = pd.to_numeric(s_norm, errors="coerce")
        if prefer_int:
            d[col] = num.round(0).astype("Int64")
        else:
            d[col] = num

    def to_text(col):
        if col in d.columns:
            d[col] = d[col].astype("string")

    def safe_drop(cols):
        for c in cols_exist(cols):
            d.drop(columns=c, inplace=True)

    # ---------------------------------------------
    # Construcción de listas de columnas de prereq
    # ---------------------------------------------
    prereq_suffixes = ["Codigo", "Nota", "Intentos", "Nivel", "Tipo", "EsCadena"]
    prereq_cols = [f"Prereq_{i}_{suf}" for i in range(1, 45) for suf in prereq_suffixes]
    prereq_cols_nota = [f"Prereq_{i}_Nota" for i in range(1, 45)]
    prereq_cols_intentos = [f"Prereq_{i}_Intentos" for i in range(1, 45)]
    prereq_cols_nivel = [f"Prereq_{i}_Nivel" for i in range(1, 45)]
    prereq_cols_tipo = [f"Prereq_{i}_Tipo" for i in range(1, 45)]
    prereq_cols_codigo = [f"Prereq_{i}_Codigo" for i in range(1, 45)]
    prereq_cols_escadena = [f"Prereq_{i}_EsCadena" for i in range(1, 45)]

    # ---------------------------------------------
    # PQ: ReplaceValue(".", ",") sobre Calificacion_Final y todas las columnas de prereq
    # ---------------------------------------------
    #replace_dot_with_comma(["Calificacion_Final"] + prereq_cols)

    # ---------------------------------------------
    # PQ: TransformColumnTypes (primer bloque)
    # Tipos base y de prereqs (hasta 17 con Int64 para Intentos/Nivel; 18-44 number)
    # ---------------------------------------------
    # Textuales
    for col in [
        "Nombre_Division", "Cod materia curso", "Materia_Aprobada", "Estado asignatura",
        "Descripcion_Materia", "DPTO Asignatura", "Nombre_Programa", "Codigo_Programa",
        "Observacion_Prerrequisito"
    ]:
        to_text(col)

    # Numéricos base
    for col in ["Periodo", "nrc", "cohorte", "Pidm"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce").astype("Int64")

    to_number("Calificacion_Final", prefer_int=False)

    # Prereq tipos:
    # Codigo/Tipo/EsCadena -> texto
    for col in cols_exist(prereq_cols_codigo + prereq_cols_tipo + prereq_cols_escadena):
        to_text(col)
    # Nota -> número
    for col in cols_exist(prereq_cols_nota):
        to_number(col, prefer_int=False)
    # Intentos/Nivel -> 1..17 Int64; 18..44 number (float)
    for i in range(1, 18):
        for col in cols_exist([f"Prereq_{i}_Intentos", f"Prereq_{i}_Nivel"]):
            to_number(col, prefer_int=True)
    for i in range(18, 45):
        for col in cols_exist([f"Prereq_{i}_Intentos", f"Prereq_{i}_Nivel"]):
            to_number(col, prefer_int=False)

    # ---------------------------------------------
    # PQ: Filtrar Cod materia curso != PML0055 y != PML7560
    # ---------------------------------------------
    if "Cod materia curso" in d.columns:
        d = d[~d["Cod materia curso"].isin(["PML0055", "PML7560"])].copy()

    # ---------------------------------------------
    # PQ: ReplaceErrorValues(Calificacion_Final, -2)
    # (en pandas: NaN por errores de parseo -> -2)
    # ---------------------------------------------
    if "Calificacion_Final" in d.columns:
        d["Calificacion_Final"] = pd.to_numeric(d["Calificacion_Final"], errors="coerce").fillna(-2)

    # ---------------------------------------------
    # PQ: Reemplazos masivos AP->5, PA->5, RP->15, NP->15 en todas las columnas de prereq
    # ---------------------------------------------
    map_reemplazos = {"AP": 5, "PA": 5, "RP": 15, "NP": 15}
    for c in cols_exist(prereq_cols):
        # reemplazar sólo strings exactos, mantener otros valores
        d[c] = d[c].replace(map_reemplazos)

    # ---------------------------------------------
    # PQ: RemoveRowsWithErrors en lista larga de prereq (emulado opcionalmente)
    # Implementación: intentamos convertir Nota/Intentos/Nivel; si no se puede y el valor no es nulo, se considera error.
    # ---------------------------------------------
    if drop_error_rows:
        err_masks = []
        for col in cols_exist(prereq_cols_nota + prereq_cols_intentos + prereq_cols_nivel):
            s = d[col]
            # Permitir NaN; error cuando to_numeric -> NaN pero había un string no nulo distinto de ''.
            s_norm = pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")
            err = s.notna() & s.astype(str).str.len().gt(0) & s_norm.isna()
            err_masks.append(err)
        if err_masks:
            any_err = np.column_stack([m.to_numpy() for m in err_masks]).any(axis=1)
            d = d.loc[~any_err].copy()

    # ---------------------------------------------
    # PQ: Join con "Asig en planes estudio" y filtrar En_plan_de_estudio == "En plan de estudio"
    # ---------------------------------------------
    if asig_en_planes_estudio is not None:
        right = asig_en_planes_estudio.copy()
        # asegurar columnas necesarias
        needed = {"MatCrso", "En_plan_de_estudio"}
        if needed.issubset(set(right.columns)) and "Cod materia curso" in d.columns:
            d = d.merge(
                right[["MatCrso", "En_plan_de_estudio"]],
                how="left",
                left_on="Cod materia curso",
                right_on="MatCrso"
            )
            # Emula Expand + filtro
            if "En_plan_de_estudio" in d.columns:
                d = d[d["En_plan_de_estudio"] == "En plan de estudio"].copy()
            # limpiar columna de join si molesta
            if "MatCrso" in d.columns:
                d.drop(columns=["MatCrso"], inplace=True)

    # ---------------------------------------------
    # PQ: Añadir periodo_pidm_nrc = Periodo & Pidm & nrc (como texto sin separadores)
    # ---------------------------------------------
    for col in ["Periodo", "Pidm", "nrc"]:
        if col in d.columns:
            d[col] = d[col].astype("Int32")
    if set(["Periodo", "Pidm", "nrc"]).issubset(d.columns):
        d["periodo_pidm_nrc"] = (
            d["Periodo"].astype("Int64").astype("string")
            + d["Pidm"].astype("Int64").astype("string")
            + d["nrc"].astype("Int64").astype("string")
        )

    # ---------------------------------------------
    # PQ: RemoveColumns (Profesor, Fecha_de_nacimiento, Nombre_Ciudad, Nombre_Pais, Nombre_Departamento)
    # ---------------------------------------------
    safe_drop(["Profesor", "Fecha_de_nacimiento", "Nombre_Ciudad", "Nombre_Pais", "Nombre_Departamento"])

    # ---------------------------------------------
    # PQ: ReplaceValue "."->"," para conjuntos específicos
    # ---------------------------------------------
    replace_dot_with_comma(["Asistencia CREE t_1"])
    replace_dot_with_comma([
        "Edad cursan asignatura", "Puntaje_icfes_recalificado", "Anio_Aplicacion_Icfes",
        "Dif anios icfes clase", "PCN_Puntaje_Ciencias_Naturales", "PIN_Puntaje_en_Ingles",
        "PLC_Puntaje_para_Lectura_Critica", "PMA_Puntaje_para_Matematicas",
        "PSC_Puntaje_Sociales_y_Ciudadanas", "Puntaje estrato"
    ])

    # ---------------------------------------------
    # PQ: TransformColumnTypes (segundo bloque) a número/int
    # ---------------------------------------------
    for col in ["pga inicial", "prom semestral t_1", "Asistencia CREE t_1", "Edad cursan asignatura",
                "Calif Final _ Retiros", "repitencia profesor referencia", "repitencia profesor referencia 1ano",
                "Puntaje estrato", "PSC_Puntaje_Sociales_y_Ciudadanas", "PMA_Puntaje_para_Matematicas",
                "PLC_Puntaje_para_Lectura_Critica", "PIN_Puntaje_en_Ingles", "PCN_Puntaje_Ciencias_Naturales",
                "Dif anios icfes clase"]:
        to_number(col, prefer_int=False)

    # 'Puntaje_icfes_recalificado' como entero (Int64) y luego eliminar 'Anio_Aplicacion_Icfes'
    to_number("Puntaje_icfes_recalificado", prefer_int=True)
    safe_drop(["Anio_Aplicacion_Icfes"])

    # ---------------------------------------------
    # PQ: Pidm a texto
    # ---------------------------------------------
    to_text("Pidm")

    # Orden opcional de columnas: mantener el orden original + nuevas al final
    # (pandas ya lo hace naturalmente salvo las añadidas)

    return d


In [3]:
## Transformacion de datos: Renombrar columnas

import pandas as pd

def renombrar_columnas(df: pd.DataFrame, conflict: str = "skip") -> pd.DataFrame:
    """
    Renombra columnas a los nombres esperados por el otro código.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame de entrada.
    conflict : {"skip","overwrite"}
        Qué hacer si el nombre destino ya existe:
        - "skip": no renombra y deja ambas (origen + destino existente), imprime aviso.
        - "overwrite": renombra y sobrescribe la columna destino.

    Retorna
    -------
    pd.DataFrame
        DataFrame con los posibles renombres aplicados.
    """
    d = df.copy()

    # Mapeos acordados (no tocar 'Periodo')
    mapping = {
        # Campos base de matrícula
        "DPTO Asignatura": "_ Matricula detalle para analisis.DPTO Asignatura",
        "Prof_Codigo": "_ Matricula detalle para analisis.Prof_Codigo",
        "pga inicial": "_ Matricula detalle para analisis.pga inicial",
        "prom semestral t_1": "_ Matricula detalle para analisis.prom semestral t_1",
        "Sexo": "_ Matricula detalle para analisis.Sexo",
        "Asistencia CREE t_1": "_ Matricula detalle para analisis.Asistencia CREE t_1",
        "Procedencia Categoria": "_ Matricula detalle para analisis.Procedencia Categoria",
        "Edad cursan asignatura": "_ Matricula detalle para analisis.Edad cursan asignatura",
        "Calif Final _ Retiros": "_ Matricula detalle para analisis.Calif Final _ Retiros",
        "repitencia profesor referencia": "_ Matricula detalle para analisis.repitencia profesor referencia",
        "repitencia profesor referencia 1ano": "_ Matricula detalle para analisis.repitencia profesor referencia 1ano",
        "Puntaje_icfes_recalificado": "_ Matricula detalle para analisis.Puntaje_icfes_recalificado",
        "Dif anios icfes clase": "_ Matricula detalle para analisis.Dif anios icfes clase",
        "PCN_Puntaje_Ciencias_Naturales": "_ Matricula detalle para analisis.PCN_Puntaje_Ciencias_Naturales",
        "PIN_Puntaje_en_Ingles": "_ Matricula detalle para analisis.PIN_Puntaje_en_Ingles",
        "PLC_Puntaje_para_Lectura_Critica": "_ Matricula detalle para analisis.PLC_Puntaje_para_Lectura_Critica",
        "PMA_Puntaje_para_Matematicas": "_ Matricula detalle para analisis.PMA_Puntaje_para_Matematicas",
        "PSC_Puntaje_Sociales_y_Ciudadanas": "_ Matricula detalle para analisis.PSC_Puntaje_Sociales_y_Ciudadanas",
        "Tipo_Colegio": "_ Matricula detalle para analisis.Tipo_Colegio",
        "Tipo_Calendario": "_ Matricula detalle para analisis.Tipo_Calendario",
        "Puntaje estrato": "_ Matricula detalle para analisis.Puntaje estrato",
        # Nota: 'Periodo' NO se toca (se solicitó mantenerlo tal cual)
        # 'eriodo' (typo) NO se corrige aquí por petición explícita
    }

    # Aplicar renombres de forma segura
    to_rename = {}
    for src, dst in mapping.items():
        if src in d.columns:
            if dst in d.columns and conflict == "skip":
                print(f"[AVISO] El destino '{dst}' ya existe. Se mantiene '{src}' sin renombrar (conflict=skip).")
                continue
            to_rename[src] = dst

    if to_rename:
        d = d.rename(columns=to_rename)

    return d


In [4]:
#Guardar resultados dataframe - > csv

import os

def guardar_resultados(df,carpeta,nombre):
        """Guarda los resultados en un archivo Excel"""
        if df is None:
            print("No hay resultados para guardar")
            return
        

        # Crear carpeta si no existe
        os.makedirs(carpeta, exist_ok=True)

        timestamp = pd.Timestamp.now().strftime("%Y-%m-%d%H%M%S")
        #nombre_archivo = "Resultados v6/resultados_prerrequisitos_v6_cadenas_"+timestamp+".xlsx"
        nombre_archivo = carpeta+nombre+timestamp+".csv"

        try:
            # Limpiar columnas completamente vacías
            df = df.dropna(axis=1, how='all')
            print(f"✓ Guardando resultados en: {nombre_archivo} ...")
            #df.to_excel(nombre_archivo, index=False)
            df.to_csv(nombre_archivo, index=False, sep=';', float_format='%.3f', decimal=',')
            print(f"✓ Resultados guardados en: {nombre_archivo}")
            print(f"Columnas en el archivo: {len(df.columns)}")
        except Exception as e:
            print(f"Error al guardar resultados: {e}")

In [5]:
# CARGAR ARCHIVO
print("=== Reivsar asignaturas. Carga Archivo Maestro.   ===\n")


# Solicitar archivo de historial
while True:
    try:
        ruta_historial = historial#input("Ingrese la ruta del archivo 'historial_asignaturas.xlsx': ").strip()
        df_historial = pd.read_csv(ruta_historial, sep=',')
        print(f"✓ Archivo de historial cargado: {len(df_historial)} registros")
        break
    except Exception as e:
        print(f"Error al cargar historial: {e}")
        print("Intente nuevamente.\n")

while True:
    try:
        ruta_asig_planes = asig_plan#input("Ingrese la ruta del archivo 'historial_asignaturas.xlsx': ").strip()
        df_asig_plan = pd.read_excel(ruta_asig_planes)
        print(f"✓ Archivo de Asignaturas en plan de estudio: {len(df_historial)} registros")
        break
    except Exception as e:
        print(f"Error al cargar historial: {e}")
        print("Intente nuevamente.\n")


=== Reivsar asignaturas. Carga Archivo Maestro.   ===

✓ Archivo de historial cargado: 690827 registros
✓ Archivo de Asignaturas en plan de estudio: 690827 registros


In [6]:
## prueba

df_test =  df_historial[df_historial['Pidm'] == 395022].copy() 
df_tes=transformar_datos(df_test, asig_en_planes_estudio=df_asig_plan, drop_error_rows=True)
df_tes= renombrar_columnas(df_tes, conflict="skip")


In [7]:
df_transformado = transformar_datos(df_historial, asig_en_planes_estudio=df_asig_plan, drop_error_rows=True)


#395022

In [8]:
d_exp= df_transformado.copy()

In [9]:
d_exp= renombrar_columnas(d_exp, conflict="skip")

In [10]:
guardar_resultados(d_exp, "Multiples Asignaturas/", "historia_todos_2019_202610_")

✓ Guardando resultados en: Multiples Asignaturas/historia_todos_2019_202610_2026-02-02181239.csv ...
✓ Resultados guardados en: Multiples Asignaturas/historia_todos_2019_202610_2026-02-02181239.csv
Columnas en el archivo: 281


In [11]:
d_exp2 = d_exp[d_exp['Periodo'] <= 202510].copy() 
guardar_resultados(d_exp2, "Multiples Asignaturas/", "historia_todos_2014_202510_")

✓ Guardando resultados en: Multiples Asignaturas/historia_todos_2014_202510_2025-10-22202332.csv ...
✓ Resultados guardados en: Multiples Asignaturas/historia_todos_2014_202510_2025-10-22202332.csv
Columnas en el archivo: 281
